# N111 · DAG动态规划与隐式偏序

**目标：** 按拓扑顺序合并状态并避免循环依赖。

**先修：** N082, N098, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有向无环；最长路；路径计数；网格递增；记忆化DFS。

## 从问题到算法

DAG上的最长路可以按拓扑序直接松弛；不需要Dijkstra，因为无环性已经给出了依赖顺序。严格递增矩阵沿数值上升连边，天然没有环。颜色路径值需保存每种颜色的最大频数，而非只保留一个当前最优颜色；不同后继可能需要不同颜色状态。

## 最小实现

**本章接口：** `longest_increasing_path(matrix)`、`largest_path_value(colors, edges)`、`minimum_course_time(n, relations, time)`

In [1]:
from collections import deque

def longest_increasing_path(matrix):
    if not matrix or not matrix[0]: return 0
    m,n=len(matrix),len(matrix[0]); indegree=[[0]*n for _ in range(m)]
    def neighbors(r,c):
        for x,y in [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]:
            if 0<=x<m and 0<=y<n: yield x,y
    for r in range(m):
        for c in range(n): indegree[r][c]=sum(matrix[x][y]<matrix[r][c] for x,y in neighbors(r,c))
    q=deque((r,c) for r in range(m) for c in range(n) if indegree[r][c]==0); layers=0
    while q:
        layers+=1
        for _ in range(len(q)):
            r,c=q.popleft()
            for x,y in neighbors(r,c):
                if matrix[x][y]>matrix[r][c]:
                    indegree[x][y]-=1
                    if indegree[x][y]==0: q.append((x,y))
    return layers

def largest_path_value(colors,edges):
    n=len(colors); adj=[[] for _ in range(n)]; degree=[0]*n
    for u,v in edges: adj[u].append(v); degree[v]+=1
    dp=[[0]*26 for _ in range(n)]; q=deque(i for i,d in enumerate(degree) if d==0); visited=best=0
    while q:
        u=q.popleft(); visited+=1; dp[u][ord(colors[u])-97]+=1; best=max(best,max(dp[u]))
        for v in adj[u]:
            for c in range(26): dp[v][c]=max(dp[v][c],dp[u][c])
            degree[v]-=1
            if degree[v]==0: q.append(v)
    return best if visited==n else -1

def minimum_course_time(n,relations,time):
    adj=[[] for _ in range(n)]; degree=[0]*n; end=list(time)
    for u,v in relations: adj[u-1].append(v-1); degree[v-1]+=1
    q=deque(i for i,d in enumerate(degree) if d==0); visited=0
    while q:
        u=q.popleft(); visited+=1
        for v in adj[u]:
            end[v]=max(end[v],end[u]+time[v]); degree[v]-=1
            if degree[v]==0: q.append(v)
    if visited!=n: raise ValueError('cyclic prerequisites')
    return max(end,default=0)

## 正确性、前提与复杂度

拓扑弹出u时所有前驱已完成，转移可安全取前驱答案的最大值。矩阵Kahn层数等于最长依赖链长度；不是到某个起点的最短距离。课程允许无限并行，开课时间取所有前置完成时间最大值；资源受限调度不是这个模型。

**代价：** 矩阵O(mn)时间空间；颜色DP O(26(V+E))时间O(26V+E)空间；课程O(V+E)时间空间。颜色限小写英文字母，课程关系使用1-based编号。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

matrix=[[9,9,4],[6,6,8],[2,1,1]]
show_table(['行','高度'],list(enumerate(matrix)))
print('严格递增最长路径',longest_increasing_path(matrix))
print('并行课程总时长',minimum_course_time(3,[(1,3),(2,3)],[3,2,5]))

行,高度
0,"[9, 9, 4]"
1,"[6, 6, 8]"
2,"[2, 1, 1]"


严格递增最长路径 4
并行课程总时长 8


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert longest_increasing_path([[9,9,4],[6,6,8],[2,1,1]])==4
assert longest_increasing_path([[7,7],[7,7]])==1
assert largest_path_value('abaca',[(0,1),(0,2),(2,3),(3,4)])==3
assert largest_path_value('a',[(0,0)])==-1
assert minimum_course_time(3,[(1,3),(2,3)],[3,2,5])==8
from random import Random
from collections import Counter
rng=Random(111)
for n in range(1,8):
    for _ in range(20):
        edges=[(u,v) for u in range(n) for v in range(u+1,n) if rng.randrange(3)==0]; colors=''.join(rng.choice('abc') for _ in range(n)); times=[rng.randrange(1,5) for _ in range(n)]
        adj=[[] for _ in range(n)]
        for u,v in edges: adj[u].append(v)
        def paths(u):
            yield [u]
            for v in adj[u]:
                for p in paths(v): yield [u]+p
        all_paths=[p for u in range(n) for p in paths(u)]
        assert largest_path_value(colors,edges)==max(max(Counter(colors[u] for u in p).values()) for p in all_paths)
        assert minimum_course_time(n,[(u+1,v+1) for u,v in edges],times)==max(sum(times[u] for u in p) for p in all_paths)
print('N111: 所有本章断言通过')

N111: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释矩阵递增约束为什么排除环。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 带环课程输入明确拒绝。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 329. Longest Increasing Path in a Matrix（canonical）
- 1857. Largest Color Value in a Directed Graph（canonical）
- 2050. Parallel Courses III（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。